# A computational solution to the problem of evaluating poker-variant hands ranking

## The definition of the problem

In poker, there is the ranking of hands, according to the probability of a hand appearing in a round.

https://en.wikipedia.org/wiki/Poker_probability

One complication is that since in some popular variations of poker such as Texas hold 'em, a player uses the best five-card poker hand out of seven cards, the ranking slighly changes.

In Greece, the similar game is called poka and is most probably related to the French version. It is usually played with 32 cards (7 to King, plus Ace), and has many variations with respect the common cards on the table and they ways they are being layed out, the cards the players get, etc. Of course this is reflected on the rankings as well.

The general ranking in poka is the following:

- straight flush
- four of a kind
- flush
- full house
- straight
- three of a kind
- two pairs
- one pair

The rule says that when the player can choose from 8 (inclusive) or more cards, or when there is a joker card, then `three of a kind` beats `straight` and `full house` beats `flush`. So the ranking becomes:

- straight flush
- four of a kind
- full house
- flush
- three of a kind
- straight
- two pairs
- one pair

The purpose here is to numerically simulate two games, one of each kind, and verify that these indeed follow the supposed rankings.


## Notes on hands

The presense of a joker card modifies the hands themselves and not just their ranking.

With a joker card, a new hand is possible: five of a kind. This hand is supposed to be the strongest of all, so there is one more hypothesis to test.

With a joker card the lowest possible hand is a pair.

Given the presense of one or more joker cards in a hand, there are multiple possibla hands. For example, with three jokers and an eight of clubs and a king of clubs, we can have a four of a kind or a flush. Depending on which one we choose the counts will change.

## Proposed solution

We are going to use straight poker as our testing ground. In this game a complete hand is being dealt to each player, without any community cards on the table. As a result, each player has access to a single combination make of the five cards of the hand - jokers apart. We strip away all the rest of the game mechanics, bets, bluffs, as these do not influnce the base probability of a hand appearing. Note that this variant is common between poker and the greek poka

We are going to develop three scenarios:

- The base scenario is the game of straight poker played with a deck of 52 cards. This will serve as acceptance test for the simulation, since we can compare the distribution of hands produced by our simulation, with the known distribution of hands: `https://en.wikipedia.org/wiki/Poker_probability`

- Then we are going to play the same with a short deck of 32 cards. We want to see have the shorter deck changes the distribution.

- Finally, we'll play straight poker, with a short deck that substitutes 7s with jokers. Again, the focus will be on how the distribution of hands changes.

Effectivelly, we have set up an experiment for comparing the base game against the two variations that modify the ranking of the hands.


### Configuration

In [1]:
"""Module docstring to keep pylint happy"""

## Imports
import itertools
import functools
import time
from collections import Counter
from typing import NamedTuple
from enum import Enum
import random
import logging
import ipytest
import pytest

## Configuration
ipytest.autoconfig()
RUN_TESTS = True

logger = logging.getLogger()
logger.setLevel(logging.INFO)
fhandler = logging.FileHandler(filename="poka.log", mode="w")
logger.addHandler(fhandler)
formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(lineno)s - %(levelname)s - %(message)s"
)
fhandler.setFormatter(formatter)


## Concepts
LONG_DECK_SIZE = 52
SHORT_DECK_SIZE = 32
## The numbers 11, 12, 13 represent the jack, queen, and king, respectivelly
LONG_DECK_RANKS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
SHORT_DECK_RANKS = [1, 7, 8, 9, 10, 11, 12, 13]
## ["Hearts", "Diamonds", "Clubs", "Spades"] -> ["H", "D", "C", "S"]
DECK_SUITS = ["H", "D", "C", "S"]
POKA_HAND_LENGTH = 5
NUM_OF_JOKERS = 4
RANK_REPLACED_BY_JOKER = 7
JOKER_TUPLE = (0, "J")

SIMULATION_LENGTH = 10000

In [2]:
def timer(func):
    """A simple decorator for timining the simulation runs"""

    @functools.wraps(func)
    def wrapper_timer(*args, **kwargs):
        tic = time.perf_counter()
        value = func(*args, **kwargs)
        toc = time.perf_counter()
        elapsed_time = toc - tic
        print(f"Elapsed time: {elapsed_time:0.4f} seconds")
        return value

    return wrapper_timer

### The deck of cards

In [3]:
## For playing poka, we need a deck of cards


class Card(NamedTuple):
    """A playing card from a deck of cards"""

    rank: int
    suit: str


class Hand(Enum):
    """The possible hands dealt"""

    HIGH = "Highest card"
    TWO = "Two of a kind"
    THREE = "Three of a kind"
    FOUR = "Four of a kind"
    FIVE = "Five of a kind"
    PAIRS = "Two pairs"
    FULL = "Full house"
    STRAIGHT = "Straight"
    FLUSH = "Flush"
    STRAIGHT_FLUSH = "Straight flush"
    UKNOWN = "I cannot tell"


## long deck
long_deck_tuples = list(itertools.product(LONG_DECK_RANKS, DECK_SUITS))
long_deck = [Card(*a_card) for a_card in long_deck_tuples]

## short deck
short_deck_tuples = list(itertools.product(SHORT_DECK_RANKS, DECK_SUITS))
short_deck = [Card(*a_card) for a_card in short_deck_tuples]

## don't need anymore
del long_deck_tuples, short_deck_tuples

## joker deck: replace each 7 with a joker card
joker_deck = [a_card for a_card in short_deck if a_card.rank != RANK_REPLACED_BY_JOKER]
jokers = [Card(*JOKER_TUPLE)] * NUM_OF_JOKERS
joker_deck = joker_deck + jokers

In [4]:
%%ipytest

if RUN_TESTS:
    
    def test_long_deck_size():
        assert len(long_deck) == LONG_DECK_SIZE, f"The deck should have exactly {LONG_DECK_SIZE} cards"

    def test_short_deck_size():
        assert len(short_deck) == SHORT_DECK_SIZE, f"The deck should have exactly {SHORT_DECK_SIZE} cards"

    def test_joker_deck_size():
        assert len(joker_deck) == SHORT_DECK_SIZE, f"The deck should have exactly {SHORT_DECK_SIZE} cards"

    def test_joker_cards_number():
        num_of_jokers = len([a_card for a_card in joker_deck if a_card.suit == Card(*JOKER_TUPLE).suit])
        assert num_of_jokers == NUM_OF_JOKERS, f"The deck should have exactly {NUM_OF_JOKERS} jokers"
        

....                                                                                         [100%]
4 passed in 0.01s


### What hand do I have?

In [5]:
def n_of_a_kind(the_hand: list[Card]) -> Hand:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
    Returns:
        one of the enum values [Hand.UKNOWN, Hand.FIVE, Hand.FOUR, Hand.THREE, Hand.TWO]
    Raises:
        ValueError: If hand does not have exactly five cards
    Notes:
        The desired combination here is n cards of the same rank
        five of a kind is only possible with a joker card deck - no joker, no 5 same ranks
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    ## Allowed values 5,4,3,2 represent combinations from five_of_a_kind to a single pair.
    allowed = {5, 4, 3, 2}

    what_hand = Hand.UKNOWN

    card_ranks = []
    for a_card in the_hand:
        card_ranks.append(a_card.rank)

    ## hand: [(1,D),(1,H),(9,H),(10,S),(10,D)] -> Counter({1: 2, 10: 2, 9: 1})
    rank_count = Counter(card_ranks)

    ## most_common(1) returns a list of (one) tupple -> [(1, 3)]
    ## where 1 is the key (rank in our case) and 3 is the count
    most_common_count = rank_count.most_common(1)[0][1]
    distinct_ranks = len(rank_count)

    for n_of in allowed:
        match n_of:
            case 5:
                ## all cards same rank - only possible with joker decks
                if most_common_count == n_of and distinct_ranks == 1:
                    what_hand = Hand.FIVE
            case 4:
                ## four cards same rank - plus another card
                if most_common_count == n_of and distinct_ranks == 2:
                    what_hand = Hand.FOUR
            case 3:
                ## three cards same rank but two more distinct ranks (otherwise it is full house)
                if most_common_count == n_of and distinct_ranks == 3:
                    what_hand = Hand.THREE
            case 2:
                ## two cards same rank but three more distinct ranks (otherwise it is full house)
                if most_common_count == n_of and distinct_ranks == 4:
                    what_hand = Hand.TWO

    return what_hand

In [6]:
%%ipytest

if RUN_TESTS:

    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            n_of_a_kind( [Card(1,'H'), 
                        Card(1,'S'),
                        Card(10,'S'), 
                        Card(10,'S')])  

    def test_long_hand():
        with pytest.raises(ValueError):
            n_of_a_kind( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')])  
   
    ###################### Assertions tests end ######################

    ###################### 5 of a kind tests start ######################
    
    def test_five_kind():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(1,'C'), 
                          Card(1,'J')]
                       ) == Hand.FIVE, f"This hand should be five of a kind"

    ###################### 5 of a kind tests end ######################
        
    ###################### 4 of a kind tests start ######################
    def test_three_plus_one():
        assert n_of_a_kind( [Card(9,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(1,'C'), 
                          Card(1,'J')]
                       ) == Hand.FOUR, f"This hand should be four of a kind"

    ###################### 4 of a kind tests end ######################
        
    ###################### 3 of a kind tests start ######################
    def test_exactly_three_kind():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(9,'C'), 
                          Card(10,'S')]
                       ) == Hand.THREE, f"This hand should be three of a kind"

    ###################### 3 of a kind tests end ###################### 
    
    ###################### 2 of a kind tests start ######################

    def test_exactly_one_pair():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(8,'D'), 
                          Card(9,'C'), 
                          Card(10,'S')]
                       ) == Hand.TWO, f"This hand should contain exactly one pair"

    ###################### 2 of a kind tests end ######################

    def test_two_pairs():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(8,'D'), 
                          Card(8,'C'), 
                          Card(10,'S')]
                       ) == Hand.UKNOWN, f"This hand is not exactly one pair"

    def test_full_house():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(8,'D'), 
                          Card(8,'C'), 
                          Card(8,'S')]
                       ) == Hand.UKNOWN, f"This hand is not exactly one pair"
    


........                                                                                     [100%]
8 passed in 0.02s


In [7]:
def flush(the_hand: list[Card]) -> Hand:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
    Returns:
        one of the enum values [Hand.UKNOWN, Hand.FLUSH]
    Raises:
        ValueError: If hand does not have exactly five cards
    Notes:
        The desired combination here is all cards of the same s
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    what_hand = Hand.UKNOWN

    card_suits = []
    for a_card in the_hand:
        card_suits.append(a_card.suit)

    ## hand: [(1,D),(1,H),(9,H),(10,S),(10,D)] -> Counter({D: 2, H: 2, S: 1})
    suit_count = Counter(card_suits)

    ## most_common(1) returns a list of (one) tupple -> [('C', 3)]
    ## where 'C' is the key (s in our case) and 3 is the count
    ## we have removed the jokers so we do not double count them, if most common
    suit_most_common_count = suit_count.most_common(1)[0][1]

    if suit_most_common_count == 5:
        what_hand = Hand.FLUSH

    return what_hand

In [8]:
%%ipytest

if RUN_TESTS:


    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            flush( [Card(1,'H'), 
                        Card(1,'S'),
                        Card(10,'S'), 
                        Card(10,'S')])  

    def test_long_hand():
        with pytest.raises(ValueError):
            flush( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')])  
    ###################### Assertions tests end ######################

    ###################### is flush tests start ######################

    def test_flush():
        assert flush( [Card(12,'H'), 
                          Card(1,'H'), 
                          Card(9,'H'), 
                          Card(10,'H'), 
                          Card(8,'H')]
                       )  == Hand.FLUSH, f"This hand is a flush"

        
    def test_no_flush():
        assert flush( [Card(12,'H'), 
                          Card(1,'H'), 
                          Card(9,'C'), 
                          Card(1,'D'), 
                          Card(8,'C')]
                       )  == Hand.UKNOWN, f"This hand is NOT a flush"

    ###################### is flush tests start ######################

....                                                                                         [100%]
4 passed in 0.01s


In [9]:
def straight_or_straigh_flush(the_hand: list[Card]) -> Hand:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
    Returns:
        one of the enum values [Hand.UKNOWN, Hand.STRAIGHT, Hand.STRAIGHT_FLUSH]
    Raises:
        ValueError: If hand does not have exactly five cards
    Notes:
        The desired combination here is all cards of sequencial rank
        An Ace can be either lowest or highest card
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    what_hand = Hand.UKNOWN

    ## An Ace can be either lowest or highest card
    straight_sequences = []
    straight_sequences.append([1, 7, 8, 9, 10])
    straight_sequences.append([7, 8, 9, 10, 11])
    straight_sequences.append([8, 9, 10, 11, 12])
    straight_sequences.append([9, 10, 11, 12, 13])
    straight_sequences.append([10, 11, 12, 13, 1])

    card_ranks = []
    card_suits = []
    for a_card in the_hand:
        card_ranks.append(a_card.rank)
        card_suits.append(a_card.suit)

    ## We start with a possible sequence and try to rebuild it with the available cards
    for straight_sequence in straight_sequences:
        sorted_hand = []
        available_cards = card_ranks.copy()

        for rank in straight_sequence:
            if rank in available_cards:
                sorted_hand.append(rank)
                available_cards.remove(rank)
            else:
                ## no need to continue if we cannot rebuilt the sequence
                break

        suit_count = len(Counter(card_suits))
        ## If it is also a flush it is a straight flush, not just straight
        if (sorted_hand in straight_sequences) and (suit_count > 1):
            what_hand = Hand.STRAIGHT
            break
        if (sorted_hand in straight_sequences) and (suit_count == 1):
            what_hand = Hand.STRAIGHT_FLUSH
            break

    return what_hand

In [10]:
%%ipytest

if RUN_TESTS:

    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            straight_or_straigh_flush( [Card(1,'H'), 
                        Card(1,'S'),
                        Card(10,'S'), 
                        Card(10,'S')])  

    def test_long_hand():
        with pytest.raises(ValueError):
            straight_or_straigh_flush( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')]) 
    ###################### Assertions tests end ######################

    ###################### is flush tests start ######################

    def test_lower_straigh():
        assert straight_or_straigh_flush( [Card(7,'H'), 
                          Card(8,'D'), 
                          Card(9,'H'), 
                          Card(10,'H'), 
                          Card(1,'H')]
                       )  == Hand.STRAIGHT, f"This hand is a straight"

    def test_upper_straight():
        assert straight_or_straigh_flush( [Card(13,'H'), 
                          Card(12,'D'), 
                          Card(11,'H'), 
                          Card(10,'H'), 
                          Card(1,'S')]
                       )  == Hand.STRAIGHT, f"This hand is a straight"
    
    def test_upper_stright_flush():
        assert straight_or_straigh_flush( [Card(13,'H'), 
                          Card(12,'H'), 
                          Card(11,'H'), 
                          Card(10,'H'), 
                          Card(1,'H')]
                       )  == Hand.STRAIGHT_FLUSH, f"This hand is NOT a straight, it is a straight flush"
    
    def test_no_straight():
        assert straight_or_straigh_flush( [Card(12,'H'), 
                          Card(1,'H'), 
                          Card(9,'H'), 
                          Card(1,''), 
                          Card(10,'')]
                       )  == Hand.UKNOWN, f"This hand is NOT a straight"
        


    ###################### is flush tests start ######################

......                                                                                       [100%]
6 passed in 0.02s


In [11]:
def combined_pairs(the_hand: list[Card]) -> Hand:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
    Returns:
        one of the enum values [Hand.UKNOWN, Hand.FULL, Hand.PAIRS]
    Raises:
        ValueError: If hand does not have exactly five cards
    Notes:
        One desired combination here is 3 cards of the same rank, plus 2 cards of nother rank
        Other desired combination here is  cards of the same rank, plus 2 cards of nother rank
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    ## Allowed values 3,2 represent combinations the highest rank.
    allowed = {3, 2}

    what_hand = Hand.UKNOWN

    card_ranks = []
    for a_card in the_hand:
        card_ranks.append(a_card.rank)

    ## hand: [(1,D),(1,H),(9,H),(10,S),(10,D)] -> Counter({1: 2, 10: 2, 9: 1})
    rank_count = Counter(card_ranks)

    ## most_common(1) returns a list of (one) tupple -> [(1, 3)]
    ## where 1 is the key (rank in our case) and 3 is the count
    most_common_count = rank_count.most_common(1)[0][1]
    distinct_ranks = len(rank_count)

    for n_of in allowed:
        match n_of:
            case 3:
                ## three cards of same rank plus two cards of another rank
                if most_common_count == n_of and distinct_ranks == 2:
                    what_hand = Hand.FULL
            case 2:
                ## two cards of same rank plus two cards of another rank
                if most_common_count == n_of and distinct_ranks == 3:
                    what_hand = Hand.PAIRS

    return what_hand

In [12]:
%%ipytest

if RUN_TESTS:


    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            combined_pairs( [Card(1,'H'), 
                        Card(1,'S'),
                        Card(10,'S'), 
                        Card(10,'S')])  

    def test_long_hand():
        with pytest.raises(ValueError):
            combined_pairs( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')]) 
    ###################### Assertions tests end ######################

    ###################### is  straight flush tests start ######################

    def test_full_house():
        assert combined_pairs( [Card(7,'H'), 
                                  Card(7,'D'), 
                                  Card(12,'H'), 
                                  Card(12,'C'), 
                                  Card(12,'S')]
                               )  == Hand.FULL, f"This hand is a full house"

    def test_two_pairs():
        assert combined_pairs( [Card(8,'S'), 
                                  Card(8,'H'), 
                                  Card(9,'S'), 
                                  Card(9,'H'),
                                  Card(12,'D')]
                               )  == Hand.PAIRS, f"This hand is a full house"
        
    def test_no_combined_pairs():
        assert combined_pairs( [Card(12,'H'), 
                              Card(1,'H'), 
                              Card(9,'C'), 
                              Card(1,'C'), 
                              Card(8,'D')]
                           )  == Hand.UKNOWN, f"This hand is NOT a full house"
        
    ###################### is  straight flush tests end ######################

.....                                                                                        [100%]
5 passed in 0.01s


In [13]:
def determine_hand(the_hand: list[Card]) -> Hand:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the s. e.g. (9, 'H')
    Returns:
        The name of the hand, drawn from an enum.
    Raises:
        ValueError: If hand does not have exactly five cards.
    Notes:
        1-Highest card hand, is not possible with a joker card; there will be at least a pair
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    what_hand = Hand.HIGH

    hand_functions = [combined_pairs, straight_or_straigh_flush, n_of_a_kind, flush]

    for hand_function in hand_functions:
        hand = hand_function(the_hand)
        if hand != Hand.UKNOWN:
            ## if we found the hand we are ready to return
            return hand

    return what_hand

In [14]:
%%ipytest

if RUN_TESTS:

    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            determine_hand( [Card(1,'H'), 
                        Card(1,'S'),
                        Card(10,'S'), 
                        Card(10,'S')])  

    def test_long_hand():
        with pytest.raises(ValueError):
            determine_hand( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')]) 
    ###################### Assertions tests end ######################

    ###################### determine hand start ######################

    def test_five_kind():
        ## Card(12, 'J') is a joker, made to match rank
        assert determine_hand( [Card(12,'J'), 
                                  Card(12,'D'), 
                                  Card(12,'H'), 
                                  Card(12,'C'), 
                                  Card(12,'S')]
                               )  == Hand.FIVE, f"This hand is five of a kind"

    def test_four_kind():
        assert determine_hand( [Card(7,'D'), 
                                  Card(12,'D'), 
                                  Card(12,'H'), 
                                  Card(12,'C'), 
                                  Card(12,'S')]
                               )  == Hand.FOUR, f"This hand is four of a kind"

    def test_three_kind():
        assert determine_hand( [Card(12,'D'), 
                                  Card(1,'D'), 
                                  Card(10,'H'), 
                                  Card(12,'C'), 
                                  Card(12,'S')]
                               )  == Hand.THREE, f"This hand is three of a kind"

    def test_two_kind():
        assert determine_hand( [Card(12,'D'), 
                                  Card(1,'D'), 
                                  Card(10,'H'), 
                                  Card(8,'C'), 
                                  Card(12,'S')]
                               )  == Hand.TWO, f"This hand is three of a kind"
        
    def test_full_house():
        assert determine_hand( [Card(7,'H'), 
                                  Card(7,'D'), 
                                  Card(12,'H'), 
                                  Card(12,'C'), 
                                  Card(12,'S')]
                               )  == Hand.FULL, f"This hand is full house"

    def test_two_pairs():
        assert determine_hand( [Card(1,'C'), 
                                  Card(1,'D'), 
                                  Card(10,'H'), 
                                  Card(12,'C'), 
                                  Card(12,'S')]
                               )  == Hand.PAIRS, f"This hand is two pairs"

    def test_straight():
        assert determine_hand( [Card(7,'C'), 
                                  Card(8,'D'), 
                                  Card(9,'C'), 
                                  Card(10,'H'), 
                                  Card(11,'S')]
                               )  == Hand.STRAIGHT, f"This hand is straight"

    def test_flush():
        assert determine_hand( [Card(7,'D'), 
                                  Card(1,'D'), 
                                  Card(9,'D'), 
                                  Card(12,'D'), 
                                  Card(11,'D')]
                               )  == Hand.FLUSH, f"This hand is flush"

    def test_straight_flush():
        assert determine_hand( [Card(8,'D'), 
                                  Card(9,'D'), 
                                  Card(10,'D'), 
                                  Card(11,'D'), 
                                  Card(12,'D')]
                               )  == Hand.STRAIGHT_FLUSH, f"This hand is straight flush"

    def test_highest_card():
        assert determine_hand( [Card(7,'D'), 
                                  Card(1,'C'), 
                                  Card(10,'D'), 
                                  Card(12,'S'), 
                                  Card(11,'D')]
                               )  == Hand.HIGH, f"This hand is highest card"

............                                                                                 [100%]
12 passed in 0.03s


### Play poker round

In [15]:
def play_poker_round(a_deck: list[Card], round_seed: int) -> list[Hand]:
    """
    Args:
        a_deck: a deck of namedtuples; each namedtuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
        seed: an integer to be used with random.seed()
    Returns:
        A list of poker hands that player 1 has
    Notes:
        We play a round of straight poker with 4 players. With the available cards each player has
        one various combination, or many when the deck has joker cards.
        We collect the combinations for player 1, and return the list
    """

    ## We need to copy the deck, because everytime we deal, we consume it
    the_deck = a_deck[:]

    random.seed(round_seed)
    random.shuffle(the_deck)

    p1: list[Card] = []
    p2: list[Card] = []
    p3: list[Card] = []
    p4: list[Card] = []

    a_game = [p1, p2, p3, p4]

    ## We deal 3 cards to each participant
    ## See intro for details on the game we deal
    for _ in range(5):
        for participant in a_game:
            try:
                participant.append(the_deck.pop())
            except IndexError as e:
                raise IndexError("The deck has no more cards") from e

    ## We now want to compute all the combinations for player1
    p1_available_cards = p1
    p1_possible_combinations = itertools.combinations(p1_available_cards, 5)

    p1_hands = []
    for dealt_hand in p1_possible_combinations:
        determined_hand = determine_hand(list(dealt_hand))
        p1_hands.append(determined_hand)
        # print(determined_hand)
        logging.info(" %s, %s", determined_hand, dealt_hand)

    return p1_hands

In [16]:
@timer
def main_simulation(sim_length: int, a_deck: list[Card], sim_seed: int) -> Counter:
    """
    Play many rounds
    """
    stats_holder: Counter = Counter()

    for i in range(sim_length):
        stats_holder.update(Counter(play_poker_round(a_deck, sim_seed + i)))
    return stats_holder

### Run simulation

In [17]:
sim_res_long = main_simulation(SIMULATION_LENGTH, long_deck, 0)

Elapsed time: 0.9394 seconds


In [18]:
sim_res_short = main_simulation(SIMULATION_LENGTH, short_deck, 0)

Elapsed time: 0.8284 seconds


### Analyse results

In [19]:
sim_res_long_sorted = sim_res_long.most_common()[::-1]
for pair in sim_res_long_sorted:
    print(pair[0], "---->", round(pair[1] / SIMULATION_LENGTH, 7))

Hand.FOUR ----> 0.0001
Hand.STRAIGHT_FLUSH ----> 0.0001
Hand.FULL ----> 0.0014
Hand.STRAIGHT ----> 0.002
Hand.FLUSH ----> 0.0028
Hand.THREE ----> 0.0195
Hand.PAIRS ----> 0.0509
Hand.TWO ----> 0.429
Hand.HIGH ----> 0.4942


In [20]:
for pair in sim_res_long_sorted:
    print(pair[0], "---->", pair[1])

Hand.FOUR ----> 1
Hand.STRAIGHT_FLUSH ----> 1
Hand.FULL ----> 14
Hand.STRAIGHT ----> 20
Hand.FLUSH ----> 28
Hand.THREE ----> 195
Hand.PAIRS ----> 509
Hand.TWO ----> 4290
Hand.HIGH ----> 4942


In [21]:
sim_res_short_sorted = sim_res_short.most_common()[::-1]
for pair in sim_res_short_sorted:
    print(pair[0], "---->", round(pair[1] / SIMULATION_LENGTH, 7))

Hand.STRAIGHT_FLUSH ----> 0.0001
Hand.FLUSH ----> 0.0006
Hand.FOUR ----> 0.0008
Hand.FULL ----> 0.0055
Hand.STRAIGHT ----> 0.0213
Hand.THREE ----> 0.0554
Hand.PAIRS ----> 0.123
Hand.HIGH ----> 0.2553
Hand.TWO ----> 0.538


In [22]:
for pair in sim_res_short_sorted:
    print(pair[0], "---->", pair[1])

Hand.STRAIGHT_FLUSH ----> 1
Hand.FLUSH ----> 6
Hand.FOUR ----> 8
Hand.FULL ----> 55
Hand.STRAIGHT ----> 213
Hand.THREE ----> 554
Hand.PAIRS ----> 1230
Hand.HIGH ----> 2553
Hand.TWO ----> 5380
